# Logistic Regression — Flight Delay Prediction

## Objective
Predict whether a flight arrives at least 15 minutes late.

## Methodology
- Use the shared leakage-safe data preparation pipeline.
- Train Logistic Regression on the training partition only.
- Evaluate on the existing chronological validation partition.
- Use Average Precision and ROC-AUC as evaluation metrics.
- Investigate the effect of regularization and class weighting.
- Do not use the held-out test partition for model selection.

In [2]:
import pandas as pd

from flight_delay.data.prepare import prepare_model_data
from flight_delay.models.logistic import fit_logistic_baseline
from flight_delay.features.model_features import (
    select_model_features,
)
from sklearn.metrics import average_precision_score, roc_auc_score

from pathlib import Path

# Locate the dataset whether the notebook runs from the
# repository root or from the notebooks/ directory.
candidates = [
    Path.cwd() / "data" / "raw" / "flights.csv",
    Path.cwd().parent / "data" / "raw" / "flights.csv",
]

DATA_PATH = next(
    (path for path in candidates if path.is_file()),
    None,
)

if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find flights.csv. Current directory: {Path.cwd()}"
    )

print("Dataset:", DATA_PATH.resolve())

splits = prepare_model_data(DATA_PATH)

X_train, y_train = splits["train"]
X_validation, y_validation = splits["validation"]

print("Training rows:", len(X_train))
print("Validation rows:", len(X_validation))
print("Training delay prevalence:", round(y_train.mean(), 4))
print("Validation delay prevalence:", round(y_validation.mean(), 4))

Dataset: /Users/ankitrajsingh/flight-delay-prediction/data/raw/flights.csv
Training rows: 228299
Validation rows: 50466
Training delay prevalence: 0.2591
Validation delay prevalence: 0.1564


In [3]:
model = fit_logistic_baseline(
    X_train,
    y_train,
    feature_set="full",
    C=1.0,
)

X_validation_selected = select_model_features(
    X_validation,
    feature_set="full",
)

validation_probabilities = model.predict_proba(
    X_validation_selected
)[:, 1]

baseline_metrics = {
    "average_precision": average_precision_score(
        y_validation, validation_probabilities
    ),
    "roc_auc": roc_auc_score(
        y_validation, validation_probabilities
    ),
}

pd.DataFrame([baseline_metrics])

,average_precision,roc_auc
0,0.203866,0.601489


In [4]:
experiment_results = []

for c_value in [0.1, 1.0, 10.0]:
    experiment_model = fit_logistic_baseline(
        X_train,
        y_train,
        feature_set="full",
        C=c_value,
    )

    probabilities = experiment_model.predict_proba(
        X_validation_selected
    )[:, 1]

    experiment_results.append({
        "C": c_value,
        "class_weight": None,
        "average_precision": average_precision_score(
            y_validation, probabilities
        ),
        "roc_auc": roc_auc_score(
            y_validation, probabilities
        ),
    })

experiment_table = pd.DataFrame(experiment_results)
experiment_table.sort_values(
    "average_precision", ascending=False
)

,C,class_weight,average_precision,roc_auc
1,1.0,None,0.203866,0.601489
2,10.0,None,0.203767,0.601226
0,0.1,None,0.203749,0.601757


In [5]:
class_weight_results = []

for weight in [None, "balanced"]:
    experiment_model = fit_logistic_baseline(
        X_train,
        y_train,
        feature_set="full",
        C=1.0,
        class_weight=weight,
    )

    probabilities = experiment_model.predict_proba(
        X_validation_selected
    )[:, 1]

    class_weight_results.append({
        "class_weight": weight if weight else "None",
        "average_precision": average_precision_score(
            y_validation, probabilities
        ),
        "roc_auc": roc_auc_score(
            y_validation, probabilities
        ),
    })

class_weight_table = pd.DataFrame(class_weight_results)
class_weight_table

,class_weight,average_precision,roc_auc
0,None,0.203866,0.601489
1,balanced,0.206450,0.605049


## Class-Imbalance Experiment — Findings

The unweighted Logistic Regression baseline achieved an Average Precision
of 0.203866 and ROC-AUC of 0.601489.

Using `class_weight="balanced"` increased Average Precision to 0.206450
and ROC-AUC to 0.605049 on the validation set.

The improvements are modest but consistent across both metrics. Balanced
class weighting is therefore worth considering for the final Logistic
Regression configuration. The result is preliminary because model
selection and comparison use the same validation set.

The standard unweighted baseline remains the reference experiment.
The held-out test partition has not been used for model selection.